# Leakage-safe features and ranking experiment

This notebook documents the full development ranking experiment. The target is order `n-1`, all retrieval and aggregate features stop at `n-2`, and `order_products__train` remains unopened. Candidate generation is frozen to the score-preserving union of Personal Repeat and Co-visitation Explore.

## Protocol

- Candidate budgets compare component K = 100, 150, and 200 before training.
- The selected pool is K = 150 per component; overlaps are deduplicated without adding unlike retrieval scores.
- One query is `(user_id, target_order_id)` and complete users are split deterministically 80/20.
- Reported nDCG and recall use the full target basket, so retrieval misses remain misses.
- The heuristic, classifier, and YetiRank model use exactly the same validation users and candidate rows.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Run from the repository root or notebooks/.')
sys.path.insert(0, str(ROOT))

from src.features import (
    ALL_FEATURES, BASE_FEATURES, CLOCK_FEATURES, INTERVAL_FEATURES,
    ITEM_FEATURES, RETRIEVAL_FEATURES, USER_FEATURES, USER_ITEM_FEATURES,
)

DATA_DIR = ROOT / 'data' / 'raw'
OUTPUT_DIR = ROOT / 'reports' / 'ranking' / 'full'
SELECTION_DIR = ROOT / 'reports' / 'ranking' / 'development_selection'
RUN_FULL_EXPERIMENT = not (OUTPUT_DIR / 'ranking_metrics.csv').exists()
OUTPUT_DIR

In [ ]:
# A clean checkout runs the real experiment once; existing immutable result
# artifacts make subsequent Restart Kernel -> Run All executions inexpensive.
if RUN_FULL_EXPERIMENT:
    command = [
        sys.executable, str(ROOT / 'scripts' / 'run_ranking_experiment.py'),
        '--data-dir', str(DATA_DIR), '--component-k', '150',
        '--sample-users', '0', '--max-neighbors', '100',
        '--batch-size', '2000', '--validation-fraction', '0.2',
        '--seed', '42', '--classifier-iterations', '120',
        '--ranker-iterations', '30', '--depth', '7',
        '--learning-rate', '0.15', '--thread-count', '10',
        '--output-dir', str(OUTPUT_DIR),
    ]
    subprocess.run(command, cwd=ROOT, check=True)

## Feature contract

Counts, shares, reorder rates, basket size, and recency are aggregated only from visible history. `ui_orders_since_last` is the distance from the prediction order to the item's last visible order; `ui_in_last_basket` marks the final visible basket. Item category ids come from product metadata. Retrieval scores remain separate and can be missing for candidates supplied by only one source. Context describes the known prediction moment, not the target contents.

In [ ]:
feature_contract = pd.DataFrame({
    'family': ['user-item', 'item', 'user', 'retrieval', 'interval context', 'clock context'],
    'features': [USER_ITEM_FEATURES, ITEM_FEATURES, USER_FEATURES,
                 RETRIEVAL_FEATURES, INTERVAL_FEATURES, CLOCK_FEATURES],
})
feature_contract.assign(features=feature_contract['features'].str.join(', '))

## Full-data artifacts and resource profile

In [ ]:
metrics = pd.read_csv(OUTPUT_DIR / 'ranking_metrics.csv')
context = pd.read_csv(OUTPUT_DIR / 'context_ablation.csv')
importance = pd.read_csv(OUTPUT_DIR / 'feature_importance.csv')
ceiling = pd.read_csv(OUTPUT_DIR / 'candidate_ceiling_validation.csv')
budget_recall = pd.read_csv(ROOT / 'reports' / 'ranking' / 'candidate_budget_recall.csv')
budget_cost = pd.read_csv(ROOT / 'reports' / 'ranking' / 'candidate_budget_cost.csv')
diagnostics = json.loads((OUTPUT_DIR / 'ranking_diagnostics.json').read_text())

assert diagnostics['run']['sample_users'] == 0
assert diagnostics['run']['all_evaluation_users'] == diagnostics['dataset']['groups']
assert diagnostics['run']['component_k'] == 150
pd.Series(diagnostics['dataset'], name='value').to_frame()

In [ ]:
candidate_budget = budget_recall.pivot(index='component_k', columns='segment', values='recall').join(
    budget_cost.set_index('component_k')[['candidate_rows', 'mean_candidates', 'pool_memory_mib']]
)
candidate_budget.round(4)

## Same-pool model comparison

In [ ]:
main_methods = ['Heuristic 2:1', 'CatBoostClassifier', 'CatBoostRanker ALL']
overall = metrics.query("method in @main_methods and segment == 'overall'").pivot(
    index='method', columns=['metric', 'k'], values='value'
).reindex(main_methods)
overall.round(4)

In [ ]:
segments = metrics.query(
    "method in @main_methods and metric == 'recall' and k == 10 and segment != 'overall'"
).pivot(index='method', columns='segment', values='value').reindex(main_methods)
segments.round(4)

## Context ablation and feature importance

In [ ]:
context.sort_values('ndcg_at_10', ascending=False).round(5)

In [ ]:
top = importance.sort_values('importance').tail(16)
ax = top.plot.barh(x='feature', y='importance', figsize=(8, 5), legend=False)
ax.set(title='CatBoostRanker ALL — PredictionValuesChange', xlabel='importance', ylabel='')
plt.tight_layout()

## Controlled convergence check and explore allocation

The single extended YetiRank run was intentionally stopped after iteration 90 when its candidate-relative validation metric reached a practical plateau. Because training snapshots were disabled, the in-memory 87-tree best model could not be recovered after interruption; no extra fit was run. The policy experiment therefore uses the frozen, fully evaluated development classifier.

In [ ]:
partial_curve = pd.read_csv(SELECTION_DIR / 'yetirank_partial_learning_curve.csv')
interruption = json.loads((SELECTION_DIR / 'yetirank_interruption.json').read_text())
policy = pd.read_csv(SELECTION_DIR / 'explore_policy_summary.csv')
policy_segments = pd.read_csv(SELECTION_DIR / 'explore_policy_segment_summary.csv')
assert interruption['reported_best_trees'] == 87
assert interruption['checkpoint_recoverable'] is False
partial_curve

In [ ]:
ax = partial_curve.plot(
    x='trees', y='candidate_relative_validation_ndcg_at_10',
    marker='o', figsize=(8, 4), legend=False,
)
ax.axvline(30, color='grey', linestyle='--', label='old 30-tree cap')
ax.axvline(87, color='tab:red', linestyle=':', label='best observed: 87 trees')
ax.set(xlabel='Trees', ylabel='Candidate-relative validation nDCG@10',
       title='Controlled YetiRank convergence check')
ax.legend()
plt.tight_layout()

In [ ]:
policy[[
    'policy', 'ndcg_at_10', 'recall_at_10',
    'repeat_recall_at_10', 'explore_recall_at_10',
    'delta_ndcg_at_10', 'delta_repeat_recall_at_10',
    'delta_explore_recall_at_10',
]].round(5)

In [ ]:
policy_segments.query("policy == '>=1 explore'")[[
    'repeat_segment', 'ndcg_at_10', 'explore_recall_at_10',
    'delta_ndcg_at_10', 'delta_explore_recall_at_10',
]].round(5)

## Conclusions

In [ ]:
def metric(method, name, k=10, segment='overall'):
    return metrics.loc[
        metrics['method'].eq(method) & metrics['metric'].eq(name)
        & metrics['k'].eq(k) & metrics['segment'].eq(segment), 'value'
    ].item()

classifier_ndcg = metric('CatBoostClassifier', 'ndcg')
ranker_ndcg = metric('CatBoostRanker ALL', 'ndcg')
heuristic_explore = metric('Heuristic 2:1', 'recall', segment='explore')
ranker_explore = metric('CatBoostRanker ALL', 'recall', segment='explore')
print(f'Classifier - heuristic nDCG@10: {classifier_ndcg - metric("Heuristic 2:1", "ndcg"):+.4f}')
print(f'Ranker - classifier nDCG@10: {ranker_ndcg - classifier_ndcg:+.4f}')
print(f'Explore Recall@10, heuristic -> 30-tree ranker: {heuristic_explore:.4f} -> {ranker_explore:.4f}')
print('The 30-tree YetiRank model loses to the classifier; the longer run plateaued near 87 trees but has no recoverable model for full-target evaluation.')
print('Selected MVP: frozen CatBoostClassifier with unconstrained score ranking; even a one-item explore quota costs 0.0100 nDCG@10.')

## Interpretation guardrails

Feature importance is descriptive, not causal. `days_since_prior_order`, `order_hour_of_day`, and `order_dow` are valid only under the stated serving assumption that the prediction moment is known. The selected candidate K is a retrieval-budget decision; the ranker still emits a short Top-10/Top-20 list. The final order `n` and `order_products__train` stay locked until the complete pipeline is frozen.